<a href="https://colab.research.google.com/github/yashul12-coder/MGB-Probe-Designer/blob/add%2Fgui-v1.6/PosNegChk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import re
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils import MeltingTemp as mt
from io import StringIO, BytesIO
from datetime import datetime
from itertools import combinations
import time
# Google Colab compatibility
try:
    import streamlit as st
    IN_COLAB = False
except:
    IN_COLAB = True

# Mock Streamlit for Colab
class MockSt:
    def __getattr__(self, name):
        return lambda *args, **kwargs: None
st = MockSt()

======================== CONFIG ========================
SPECIES_DB = {
    "Human (GRCh38)": {
        "ensembl": "homo_sapiens",
        "assembly": "GRCh38",
        "taxid": "9606"
    },
    "Mouse (GRCm39)": {
        "ensembl": "mus_musculus",
        "assembly": "GRCm39",
        "taxid": "10090"
    },
    "An. gambiae": {
        "ensembl": "anopheles_gambiae",
        "assembly": "AgamP4",
        "taxid": "7165"
    },
    "Zebrafish": {
        "ensembl": "danio_rerio",
        "assembly": "GRCz11",
        "taxid": "7955"
    },
    "Arabidopsis": {
        "ensembl": "arabidopsis_thaliana",
        "assembly": "TAIR10",
        "taxid": "3702"
    }
}

FLUOR_PAIRS = {
    "FAM": {"em": 520, "quencher": "NFQ-MGB", "color": "🟢"},
    "VIC": {"em": 554, "quencher": "NFQ-MGB", "color": "🟡"},
    "ABY": {"em": 580, "quencher": "NFQ-MGB", "color": "🟠"},
    "JUN": {"em": 610, "quencher": "BHQ2-MGB", "color": "🔴"},
}

======================== CORE FUNCTIONS ========================
def fetch_rsid_data(rsid, species="homo_sapiens", flank=300):
    """Fetch SNP data from Ensembl REST API with retry logic"""
    server = "https://rest.ensembl.org"
    for attempt in range(3):
        try:
            # Get variant info
            ext = f"/variation/{species}/{rsid}?content-type=application/json"
            r = requests.get(server + ext, headers={"Content-Type": "application/json"}, timeout=10)

            if not r.ok:
                if attempt < 2:
                    time.sleep(1)
                    continue
                return {"error": f"HTTP {r.status_code}: {r.reason}"}

            data = r.json()
            mappings = data.get("mappings", [])

            if not mappings:
                return {"error": "No genomic mappings found for this RSID in the specified species."}

            mapping = mappings[0]
            chrom = mapping["seq_region_name"]
            start = mapping["start"]
            end = mapping["end"]

            # Get flanking sequence
            seq_ext = f"/sequence/region/{species}/{chrom}:{start-flank}..{end+flank}?content-type=application/json"
            seq_r = requests.get(server + seq_ext, timeout=10)

            if not seq_r.ok:
                return {"error": f"Failed to fetch sequence: {seq_r.status_code}"}

            seq_data = seq_r.json()
            sequence = seq_data["seq"].upper()

            # Parse alleles
            alleles = data.get("alleles", [])
            ancestral = data.get("ancestral_allele", alleles[0] if alleles else "N")

            # Filter to valid alleles
            valid_alleles = [a for a in alleles if a in "ACGT"]

            if not valid_alleles:
                return {"error": "No valid nucleotide alleles (A,C,G,T) found for this RSID."}

            return {
                "rsid": data.get("name", rsid),
                "sequence": sequence,
                "snp_pos": flank,
                "ref_allele": ancestral if ancestral in "ACGT" else valid_alleles[0],
                "alleles": valid_alleles,
                "location": f"{chrom}:{start}-{end}",
                "minor_allele": data.get("minor_allele"),
                "maf": data.get("minor_allele_freq")
            }

        except requests.Timeout:
            if attempt < 2:
                time.sleep(1)
                continue
            return {"error": "Request to Ensembl timed out after multiple attempts. Server might be slow or unreachable."}
        except Exception as e:
            return {"error": f"An unexpected error occurred while fetching data: {str(e)}"}

    return {"error": "Failed to fetch data after multiple attempts due to unhandled issues."}

def calculate_tm(seq):
    """Calculate melting temperature"""
    try:
        return mt.Tm_NN(seq, Na=50, K=0, Tris=0, Mg=2, dNTPs=0.8)
    except ValueError: # Catch specific error for invalid sequence input
        # Fallback to basic formula if Biopython's Tm_NN fails
        return 2 * (seq.count('A') + seq.count('T')) + 4 * (seq.count('G') + seq.count('C'))
def calculate_mgb_tm(seq):
    """Accurate MGB Tm calculation"""
    base_tm = calculate_tm(seq)
    length = len(seq)
    if length <= 15:
        mgb_boost = 10
    elif length <= 18:
        mgb_boost = 7
    else:
        mgb_boost = 5

    return base_tm + mgb_boost
def get_destabilizing_mismatch(base):
    """Get mismatch base for ARMS primer - maximally destabilizing"""
    mismatch_map = {'A': 'C', 'T': 'G', 'G': 'T', 'C': 'A'}
    return mismatch_map.get(base, 'N')
def check_primer_quality(primer_seq):
    """Quick local validation checks"""
    issues = []
    # Check for repeats (4+ same base)
    if any(base * 4 in primer_seq for base in 'ACGT'):
        issues.append("⚠️ Contains 4+ base repeat")

    # Check for self-complementarity
    rev_comp = str(Seq(primer_seq).reverse_complement())
    for i in range(len(primer_seq) - 5):
        if primer_seq[i:i+6] in rev_comp:
            issues.append("⚠️ High self-complementarity")
            break

    # Check 3' stability
    gc_3prime = primer_seq[-5:].count('G') + primer_seq[-5:].count('C')
    if gc_3prime > 4:
        issues.append("⚠️ Too stable at 3' end (GC > 4/5)")
    elif gc_3prime < 1:
        issues.append("⚠️ Too weak at 3' end (GC < 1/5)")

    # Check for runs at 3' end
    if any(base * 3 in primer_seq[-6:] for base in 'ACGT'):
        issues.append("⚠️ 3+ base run near 3' end")

    return issues
def check_primer_dimer(seq1, seq2):
    """Check for primer dimer formation"""
    rev_comp2 = str(Seq(seq2).reverse_complement())
    # Check 3' complementarity (most critical 8 bases)
    end1 = seq1[-8:]
    end2_rc = rev_comp2[-8:]

    matches = sum(a == b for a, b in zip(end1, end2_rc))

    if matches >= 6:
        return True, f"High 3' complementarity ({matches}/8 bp)"

    # Check for longer internal complementarity
    for i in range(len(seq1) - 9):
        window = seq1[i:i+10]
        if window in rev_comp2:
            return True, f"10 bp complementarity found"

    return False, "OK"
def design_arms_primer(seq, snp_pos, allele, mismatch_pos=-3, target_tm=(58, 62)):
    """Design ARMS primer with FIXED indexing"""
    best_primer = None
    best_score = float('inf')
    for length in range(20, 28):
        # Calculate start position so SNP is near 3' end
        # For mismatch_pos=-3, SNP should be at position length-3
        start_pos = snp_pos - length + abs(mismatch_pos) + 1

        if start_pos < 0 or start_pos + length > len(seq):
            continue

        # Extract primer sequence as list for easy modification
        primer = list(seq[start_pos:start_pos + length])

        # Calculate SNP position within primer
        snp_in_primer = snp_pos - start_pos

        # Insert specific allele at SNP position
        primer[snp_in_primer] = allele

        # Add destabilizing mismatch at correct position
        # mismatch_pos=-3 means 3 bases from 3' end (index -3)
        # mismatch_pos=-2 means 2 bases from 3' end (index -2)
        mm_idx = len(primer) + mismatch_pos

        if 0 <= mm_idx < len(primer) and mm_idx != snp_in_primer:
            original_base = primer[mm_idx]
            primer[mm_idx] = get_destabilizing_mismatch(original_base)

        primer = ''.join(primer)

        # Validate primer
        tm = calculate_tm(primer)
        gc = (primer.count('G') + primer.count('C')) / len(primer) * 100

        # Quality filters
        if not (30 < gc < 70):
            continue
        if primer[-4:].count('G') + primer[-4:].count('C') > 3:
            continue

        # Check for issues
        issues = check_primer_quality(primer)
        if any("repeat" in i.lower() for i in issues):
            continue

        # Score based on Tm target
        score = abs(tm - (target_tm[0] + target_tm[1]) / 2)

        if score < best_score and target_tm[0] <= tm <= target_tm[1]:
            best_score = score
            best_primer = {
                "seq": primer,
                "tm": round(tm, 1),
                "gc": round(gc, 1),
                "issues": issues,
                "snp_pos": snp_in_primer,
                "mismatch_pos": mm_idx
            }

    return best_primer
def design_mgb_probe(seq, snp_pos, allele, target_tm=(66, 70), length_range=(15, 18)):
    """Design MGB probe with allele discrimination"""
    best_probe = None
    best_score = float('inf')
    for length in range(length_range[0], length_range[1] + 1):
        # Try different SNP positions within probe
        for snp_offset in range(3, length - 3):  # SNP not too close to ends
            start_pos = snp_pos - snp_offset

            if start_pos < 0 or start_pos + length > len(seq):
                continue

            probe = list(seq[start_pos:start_pos + length])
            probe[snp_offset] = allele
            probe = ''.join(probe)

            # MGB probe rules
            if probe[0] == 'G':  # No 5' G (quenching issues)
                continue
            if probe[-1] not in ['G', 'C']:  # Need 3' G or C
                continue

            gc = (probe.count('G') + probe.count('C')) / len(probe) * 100
            if gc < 40 or gc > 65:
                continue

            # No G runs (quenching)
            if 'GGGG' in probe:
                continue

            tm = calculate_mgb_tm(probe)
            score = abs(tm - (target_tm[0] + target_tm[1]) / 2)

            if score < best_score and target_tm[0] <= tm <= target_tm[1]:
                best_score = score
                best_probe = {
                    "seq": probe,
                    "tm": round(tm, 1),
                    "gc": round(gc, 1),
                    "snp_pos": snp_offset
                }

    return best_probe
def design_common_primer(seq, snp_pos, target_tm=(58, 62), direction='reverse'):
    """Design common primer (non-allele-specific)"""
    best_primer = None
    best_score = float('inf')
    for length in range(20, 28):
        if direction == 'reverse':
            start_pos = snp_pos + 60  # 60 bp downstream of SNP
        else:
            start_pos = snp_pos - length - 60  # 60 bp upstream

        if start_pos < 0 or start_pos + length > len(seq):
            continue

        primer = seq[start_pos:start_pos + length]

        if direction == 'reverse':
            primer = str(Seq(primer).reverse_complement())

        tm = calculate_tm(primer)
        gc = (primer.count('G') + primer.count('C')) / len(primer) * 100

        if not (30 < gc < 70):
            continue

        issues = check_primer_quality(primer)
        if any("repeat" in i.lower() for i in issues):
            continue

        score = abs(tm - (target_tm[0] + target_tm[1]) / 2)

        if score < best_score and target_tm[0] <= tm <= target_tm[1]:
            best_score = score
            best_primer = {
                "seq": primer,
                "tm": round(tm, 1),
                "gc": round(gc, 1),
                "issues": issues
            }

    return best_primer
def extract_amplicon(seq, snp_pos, allele, flank=150, restriction_site_5prime="GAATTC", restriction_site_3prime="AAGCTT"):
    """Extract amplicon for G-block with cloning sites"""
    start = max(0, snp_pos - flank)
    end = min(len(seq), snp_pos + flank + 1)
    core = seq[start:snp_pos] + allele + seq[snp_pos+1:end]

    # Add flanking restriction sites for cloning
    gblock = restriction_site_5prime + core + restriction_site_3prime

    return {
        "sequence": gblock,
        "core_length": len(core),
        "total_length": len(gblock),
        "restriction_sites": f"{restriction_site_5prime} (5') / {restriction_site_3prime} (3')"
    }
def generate_genotype_interpretation(assays, snp_name, alleles):
    """Generate comprehensive result interpretation table"""
    # Create all possible genotype combinations
    genotypes = []
    for a1 in alleles:
        for a2 in alleles:
            if (a1, a2) not in [(g['a1'], g['a2']) for g in genotypes] and \
               (a2, a1) not in [(g['a1'], g['a2']) for g in genotypes]:
                genotypes.append({
                    'genotype': f"{a1}/{a2}" if a1 <= a2 else f"{a2}/{a1}",
                    'a1': a1 if a1 <= a2 else a2,
                    'a2': a2 if a1 <= a2 else a1,
                    'type': 'Homozygous' if a1 == a2 else 'Heterozygous'
                })

    # Create interpretation table
    interpretation = []

    for geno in genotypes:
        # Determine which assays should amplify
        expected_signals = []

        for assay in assays:
            if assay['allele_type'] in ['REF', 'ALT']:
                allele = assay['allele']
                # Assay amplifies if genotype contains this allele
                if allele in [geno['a1'], geno['a2']]:
                    expected_signals.append(allele)

        # Format expected results
        if len(expected_signals) == 1:
            result = f"{expected_signals[0]} only"
        elif len(expected_signals) == 2:
            result = f"{expected_signals[0]} + {expected_signals[1]}"
        else:
            result = "No signal"

        interpretation.append({
            'Genotype': geno['genotype'],
            'Zygosity': geno['type'],
            'Expected_Signal': result,
            'Description': get_genotype_description(snp_name, geno['genotype'])
        })

    return pd.DataFrame(interpretation)
def get_genotype_description(snp_name, genotype):
    """Get clinical/functional description for common SNPs"""
    descriptions = {
        'rs671': {
            'G/G': 'Normal ALDH2 - efficient alcohol metabolism',
            'G/A': 'Heterozygous - reduced ALDH2 activity (~50%)',
            'A/A': 'Inactive ALDH2 - severe alcohol intolerance'
        },
        'rs738409': {
            'C/C': 'Normal PNPLA3 - lower NAFLD risk',
            'C/G': 'Heterozygous - intermediate risk',
            'G/G': 'I148M variant - higher NAFLD/cirrhosis risk'
        },
        'rs6025': {
            'G/G': 'Normal Factor V - no thrombophilia',
            'G/A': 'Factor V Leiden heterozygous - 5-7x clot risk',
            'A/A': 'Factor V Leiden homozygous - 50-100x clot risk'
        }
    }

    if snp_name in descriptions and genotype in descriptions[snp_name]:
        return descriptions[snp_name][genotype]

    return "See literature for phenotype association"
def check_multiplex_compatibility(assays):
    """Check if assays can be multiplexed"""
    issues = []
    primers = [a['primer_forward'] for a in assays if a.get('primer_forward')] # Assuming primer_forward is the 'primer' for checks
    probes = [a['probe'] for a in assays if a.get('probe')]

    # Check Tm compatibility
    primer_tms = [p['tm'] for p in primers]
    if primer_tms and (max(primer_tms) - min(primer_tms) > 2):
        issues.append(f"⚠️ Primer Tm range: {min(primer_tms)}-{max(primer_tms)}°C (>2°C difference)")

    probe_tms = [p['tm'] for p in probes]
    if probe_tms and (max(probe_tms) - min(probe_tms) > 3):
        issues.append(f"⚠️ Probe Tm range: {min(probe_tms)}-{max(probe_tms)}°C (>3°C difference)")

    # Check primer dimers
    primer_seqs = [p['seq'] for p in primers]
    # Also include reverse primers if applicable for dimer checks
    reverse_primers = [a['primer_reverse'] for a in assays if a.get('primer_reverse')]
    primer_seqs.extend([p['seq'] for p in reverse_primers if p]) # Add reverse primers to the list

    for i, p1 in enumerate(primer_seqs):
        for j, p2 in enumerate(primer_seqs):
            if i >= j: continue # Avoid duplicate checks and self-comparison
            has_dimer, msg = check_primer_dimer(p1, p2)
            if has_dimer:
                issues.append(f"⚠️ Primer pair ({i+1}, {j+1}): {msg}")

    if not issues:
        return True, "✅ All assays compatible for multiplexing"

    return False, " | ".join(issues)
def design_assay(name, sequence, snp_pos, ref_allele, all_alleles, params):
    """Design complete assay for all alleles"""
    assays = []
    # Design common reverse primer
    common_primer = design_common_primer(sequence, snp_pos, params['primer_tm'])

    for allele in all_alleles:
        allele_type = "REF" if allele == ref_allele else "ALT"

        # Create sequence with this allele
        allele_seq = sequence[:snp_pos] + allele + sequence[snp_pos+1:]

        primer = design_arms_primer(
            allele_seq, snp_pos, allele,
            params['mismatch_pos'],
            params['primer_tm']
        )

        probe = design_mgb_probe(
            allele_seq, snp_pos, allele,
            params['probe_tm'],
            params['probe_length']
        )

        if not primer or not probe:
            continue

        amplicon_data = extract_amplicon(allele_seq, snp_pos, allele)

        assays.append({
            "name": name,
            "allele": allele,
            "allele_type": allele_type,
            "primer_forward": primer,
            "primer_reverse": common_primer,
            "probe": probe,
            "amplicon": amplicon_data,
            "sequence": allele_seq,
            "snp_pos": snp_pos
        })

    # Add WT control if requested
    if params.get('include_wt'):
        wt_amplicon = extract_amplicon(sequence, snp_pos, ref_allele, flank=250)
        assays.append({
            "name": name,
            "allele": "WT",
            "allele_type": "WT_CONTROL",
            "primer_forward": None,
            "primer_reverse": None,
            "probe": None,
            "amplicon": wt_amplicon,
            "sequence": sequence,
            "snp_pos": snp_pos
        })

    return assays